In [1]:
import os
import re
import string
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from underthesea import word_tokenize

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

In [2]:
DATA_FILE = "../output/news_labeled_clean.csv"
df = pd.read_csv(DATA_FILE)
df = df.sample(n=50_000, random_state=42).reset_index(drop=True)
print(df.shape)
print(df.columns.tolist())
df.head(3)

(50000, 10)
['id', 'url', 'category', 'label', 'title', 'desc', 'text', 'source', 'author', 'crawled_at']


,id,url,category,label,title,desc,text,source,author,crawled_at
0,103003,https://tienphong.vn/bi-thu-t-u-doan-tang-qua-doi-hinh-tiep-suc-mua-thi-vinh-phuc-ninh-binh-post1451781.tpo,giới trẻ,0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...",tienphong,Xuân Tùng,2022-07-07 14:52:37.891739
1,13977,https://www.24h.com.vn/ban-tre-cuoc-song/do-khoc-do-cuoi-voi-phan-ung-cua-be-gai-khi-bai-kiem-tra-bi-cho-cung-can-nat-c64a1369635.html,bạn trẻ - cuộc sống,0,Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát,"Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...","Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...",24h.com.vn,Theo Đinh Kim (T/h) (Đời sống & Pháp luật),2022-06-17 17:24:44.587671
2,62431,https://vov.vn/xa-hoi/tin-24h/hai-hoc-sinh-duoi-nuoc-khi-tam-bien-o-quang-tri-post953629.vov,xã hội,0,Hai học sinh đuối nước khi tắm biển ở Quảng Trị,"Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị đuối nước. Một học sinh bơi được vào bờ và tri ...","Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị đuối nước. Một học sinh bơi được vào bờ và tri ...",vov.vn,Đình Thiệu/VOV-Miền Trung,2022-06-30 13:04:53.246926


In [3]:
df["title_raw"] = df["title"].fillna("").astype(str)
df["desc_raw"] = df["desc"].fillna("").astype(str)
df["text_raw"] = df["text"].fillna("").astype(str)

df[["title_raw", "desc_raw", "text_raw"]].head(2)

,title_raw,desc_raw,text_raw
0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T..."
1,Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát,"Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...","Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g..."


In [4]:
df["model_input_raw"] = (
    df["title_raw"].str.strip() + " " + df["text_raw"].str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

df[["title_raw", "text_raw", "model_input_raw"]].head(2)

,title_raw,text_raw,model_input_raw
0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...","Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hìn..."
1,Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát,"Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...","Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bê..."


In [5]:
vi_stopwords = {
    "và", "là", "của", "có", "được", "trong", "một", "những", "các", "cho", "với",
    "khi", "đã", "đang", "theo", "về", "ở", "ra", "này", "đó", "thì", "lại", "từ",
    "đến", "sau", "trên", "dưới", "tại", "bị", "do", "nên", "vẫn", "rằng", "như",
    "để", "hay", "vào", "hơn", "ít", "nhiều", "rất", "cũng", "mới"
}

In [6]:
def preprocess_vietnamese_text(text):
    text = str(text).lower().strip()
    
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"[“”‘’…–—]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    
    tokens = text.split()
    tokens = [tok for tok in tokens if tok not in vi_stopwords]
    text = " ".join(tokens)
    
    return text

df["model_input_clean"] = df["model_input_raw"].apply(
    lambda x: preprocess_vietnamese_text(x)
)

df[["model_input_raw", "model_input_clean"]].head(3)

,model_input_raw,model_input_clean
0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hìn...",bí thư t ư đoàn tặng quà đội hình tiếp sức mùa thi vĩnh phúc ninh bình vĩnh phúc đoàn công tác anh nguyễn tường lâm bí thư t ư đoàn làm trưởng đoàn thăm tặng quà động viên đội hình thanh niên tình...
1,"Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bê...",dở khóc dở cười phản ứng bé gái bài kiểm tra chó cưng cắn nát bà mẹ yên đài tỉnh sơn đông trung quốc đây đăng tải đoạn video ghi cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng tạo hai ...
2,"Hai học sinh đuối nước khi tắm biển ở Quảng Trị Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị...",hai học sinh đuối nước tắm biển quảng trị trước cuối buổi chiều nhóm học sinh xã phong bình huyện gio linh rủ nhau tắm biển khu vực thôn xã gio hải huyện gio linh đuối nước học sinh bơi bờ tri hô ...


In [7]:
X = df["model_input_clean"]
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))
print("Train label distribution:")
print(y_train.value_counts(normalize=True).sort_index())
print("Test label distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Train size: 40000
Test size : 10000
Train label distribution:
label
0    0.910175
1    0.089825
Name: proportion, dtype: float64
Test label distribution:
label
0    0.9102
1    0.0898
Name: proportion, dtype: float64


# Hướng H4 — Thử N-gram

In [8]:
vectorizers_bigram = {
    "BoW":    CountVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95, max_features=50000),
    "TF-IDF": TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.95, max_features=50000),
}

models_h4 = {
    "MultinomialNB":      MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM":                LinearSVC(random_state=42),
}

results_h4 = []

for vec_name, vectorizer in vectorizers_bigram.items():
    for model_name, model in models_h4.items():
        pipeline = Pipeline([("vectorizer", vectorizer), ("classifier", model)])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average="binary", pos_label=1, zero_division=0
        )

        results_h4.append({
            "pipeline":          f"{vec_name} (1,2) + {model_name}",
            "vectorizer":        vec_name,
            "model":             model_name,
            "ngram":             "(1,2)",
            "accuracy":          round(acc, 4),
            "precision_class_1": round(precision, 4),
            "recall_class_1":    round(recall, 4),
            "f1_class_1":        round(f1, 4),
        })

results_h4_df = pd.DataFrame(results_h4).sort_values(
    by=["f1_class_1", "accuracy"], ascending=False
).reset_index(drop=True)

print("=== Kết quả sau cải tiến H4 (Bigram) ===")
results_h4_df

=== Kết quả sau cải tiến H4 (Bigram) ===


,pipeline,vectorizer,model,ngram,accuracy,precision_class_1,recall_class_1,f1_class_1
0,"TF-IDF (1,2) + SVM",TF-IDF,SVM,"(1,2)",0.9478,0.7467,0.6336,0.6855
1,"TF-IDF (1,2) + LogisticRegression",TF-IDF,LogisticRegression,"(1,2)",0.9478,0.7733,0.5924,0.6709
2,"TF-IDF (1,2) + MultinomialNB",TF-IDF,MultinomialNB,"(1,2)",0.9402,0.6674,0.6659,0.6667
3,"BoW (1,2) + LogisticRegression",BoW,LogisticRegression,"(1,2)",0.9392,0.6730,0.6281,0.6498
4,"BoW (1,2) + SVM",BoW,SVM,"(1,2)",0.9303,0.6120,0.6114,0.6117
5,"BoW (1,2) + MultinomialNB",BoW,MultinomialNB,"(1,2)",0.8910,0.4475,0.9120,0.6004


In [9]:
vectorizers_baseline = {
    "BoW":    CountVectorizer(ngram_range=(1, 1), min_df=3, max_df=0.95),
    "TF-IDF": TfidfVectorizer(ngram_range=(1, 1), min_df=3, max_df=0.95),
}
models_baseline = {
    "MultinomialNB":      MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM":                LinearSVC(random_state=42),
}
results = []
for vec_name, vectorizer in vectorizers_baseline.items():
    for model_name, model in models_baseline.items():
        pipeline = Pipeline([("vectorizer", vectorizer), ("classifier", model)])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average="binary", pos_label=1, zero_division=0
        )
        results.append({
            "vectorizer": vec_name, "model": model_name,
            "accuracy": round(acc, 4), "precision_class_1": round(precision, 4),
            "recall_class_1": round(recall, 4), "f1_class_1": round(f1, 4),
        })
results_df = pd.DataFrame(results).sort_values(
    by=["f1_class_1", "accuracy"], ascending=False
).reset_index(drop=True)
results_df

,vectorizer,model,accuracy,precision_class_1,recall_class_1,f1_class_1
0,TF-IDF,SVM,0.9451,0.7293,0.6180,0.6691
1,TF-IDF,LogisticRegression,0.9435,0.7467,0.5612,0.6408
2,BoW,LogisticRegression,0.9303,0.6110,0.6158,0.6134
3,BoW,MultinomialNB,0.8870,0.4375,0.9042,0.5897
4,BoW,SVM,0.9224,0.5624,0.6125,0.5864
5,TF-IDF,MultinomialNB,0.9328,0.8087,0.3296,0.4684


In [10]:
# ── H4.3: So sánh kết quả Unigram (1,1) vs Bigram (1,2) ─────────────────────

# Gắn nhãn ngram cho baseline
baseline_ngram = results_df[["vectorizer", "model", "accuracy",
                              "precision_class_1", "recall_class_1", "f1_class_1"]].copy()
baseline_ngram["ngram"] = "(1,1)"

# Gắn nhãn ngram cho bigram
after_ngram = results_h4_df[["vectorizer", "model", "accuracy",
                               "precision_class_1", "recall_class_1", "f1_class_1"]].copy()
after_ngram["ngram"] = "(1,2)"

# Gộp và pivot để xem side-by-side
all_ngram = pd.concat([baseline_ngram, after_ngram], ignore_index=True)

compare_h4 = all_ngram.pivot_table(
    index=["vectorizer", "model"],
    columns="ngram",
    values=["accuracy", "f1_class_1", "recall_class_1", "precision_class_1"]
).round(4)

compare_h4.columns = ["_".join(col) for col in compare_h4.columns]
compare_h4 = compare_h4.reset_index()

# Tính delta F1
compare_h4["f1_delta"] = (compare_h4["f1_class_1_(1,2)"] - compare_h4["f1_class_1_(1,1)"]).round(4)

print("=== So sánh Unigram (1,1) vs Bigram (1,2) ===")
compare_h4[["vectorizer", "model",
            "f1_class_1_(1,1)", "f1_class_1_(1,2)", "f1_delta",
            "accuracy_(1,1)",   "accuracy_(1,2)"]].sort_values("f1_delta", ascending=False)


=== So sánh Unigram (1,1) vs Bigram (1,2) ===


,vectorizer,model,"f1_class_1_(1,1)","f1_class_1_(1,2)",f1_delta,"accuracy_(1,1)","accuracy_(1,2)"
4,TF-IDF,MultinomialNB,0.4684,0.6667,0.1983,0.9328,0.9402
0,BoW,LogisticRegression,0.6134,0.6498,0.0364,0.9303,0.9392
3,TF-IDF,LogisticRegression,0.6408,0.6709,0.0301,0.9435,0.9478
2,BoW,SVM,0.5864,0.6117,0.0253,0.9224,0.9303
5,TF-IDF,SVM,0.6691,0.6855,0.0164,0.9451,0.9478
1,BoW,MultinomialNB,0.5897,0.6004,0.0107,0.8870,0.8910


# Hướng H7 — Xử lý từ tiếng Việt

In [11]:
def segment_series(text_series: pd.Series) -> pd.Series:
    """Áp dụng word_tokenize cho từng dòng, trả về Series đã tách từ."""
    tqdm.pandas(desc="Word segmentation")
    return text_series.progress_apply(
        lambda text: word_tokenize(str(text), format="text")
    )

# Kiểm tra nhanh
sample = "Chàng trai 9X Quảng Trị khởi nghiệp từ nấm sò"
print("Trước:", sample)
print("Sau  :", word_tokenize(sample, format="text"))

Trước: Chàng trai 9X Quảng Trị khởi nghiệp từ nấm sò
Sau  : Chàng trai 9X Quảng_Trị khởi_nghiệp từ nấm sò


In [12]:
print("Đang tách từ X_train...")
X_train_seg = segment_series(X_train.reset_index(drop=True))

print("\nĐang tách từ X_test...")
X_test_seg = segment_series(X_test.reset_index(drop=True))

print("\n--- Ví dụ so sánh trước/sau ---")
for i in range(3):
    print(f"\n[{i}] Gốc : {X_train.iloc[i][:80]}...")
    print(f"[{i}] Seg : {X_train_seg.iloc[i][:80]}...")

Đang tách từ X_train...


Word segmentation: 100%|██████████| 40000/40000 [19:14<00:00, 34.64it/s]



Đang tách từ X_test...


Word segmentation: 100%|██████████| 10000/10000 [04:52<00:00, 34.21it/s]


--- Ví dụ so sánh trước/sau ---

[0] Gốc : giúp hai mẹ con quên giấy tờ sân bay tài xế buồn lòng vì ngờ vực báo dân trí tài...
[0] Seg : giúp hai mẹ_con quên giấy_tờ sân_bay tài_xế buồn_lòng vì ngờ_vực báo dân_trí tài...

[1] Gốc : phát hiện đồ trang trí bằng xương người năm tuổi người tiền sử báo dân trí con n...
[1] Seg : phát_hiện đồ trang_trí bằng xương người năm_tuổi người tiền sử_báo dân_trí con_n...

[2] Gốc : chồng đoàn di băng bất ngờ nói chuyện ngoại tình hút bão like trang cá nhân doan...
[2] Seg : chồng đoàn di_băng bất_ngờ nói_chuyện ngoại_tình hút bão like_trang cá_nhân doan...


In [13]:
cache_path = "segmented_cache.csv"

if not os.path.exists(cache_path):
    cache_df = pd.DataFrame({
        "index": X_train.index.tolist() + X_test.index.tolist(),
        "split": ["train"] * len(X_train_seg) + ["test"] * len(X_test_seg),
        "text_seg": X_train_seg.tolist() + X_test_seg.tolist(),
    })
    cache_df.to_csv(cache_path, index=False)
    print(f"Đã lưu cache → {cache_path}")
else:
    print(f"Cache đã tồn tại ({cache_path}), load lại...")
    cache_df = pd.read_csv(cache_path)
    X_train_seg = pd.Series(cache_df[cache_df["split"] == "train"]["text_seg"].values)
    X_test_seg  = pd.Series(cache_df[cache_df["split"] == "test" ]["text_seg"].values)
    print("Load cache xong.")

Đã lưu cache → segmented_cache.csv


In [14]:
# Dùng cùng config vectorizer với baseline
vectorizers_h7 = {
    "BoW":    CountVectorizer(ngram_range=(1, 1), min_df=3, max_df=0.95),
    "TF-IDF": TfidfVectorizer(ngram_range=(1, 1), min_df=3, max_df=0.95),
}

models_h7 = {
    "MultinomialNB":      MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM":                LinearSVC(random_state=42),
}

results_h7 = []

for vec_name, vectorizer in vectorizers_h7.items():
    for model_name, model in models_h7.items():
        pipeline = Pipeline([("vectorizer", vectorizer), ("classifier", model)])
        pipeline.fit(X_train_seg, y_train)
        y_pred = pipeline.predict(X_test_seg)

        acc = accuracy_score(y_test, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average="binary", pos_label=1, zero_division=0
        )

        results_h7.append({
            "pipeline":          f"{vec_name} + {model_name} [segmented]",
            "vectorizer":        vec_name,
            "model":             model_name,
            "accuracy":          round(acc, 4),
            "precision_class_1": round(precision, 4),
            "recall_class_1":    round(recall, 4),
            "f1_class_1":        round(f1, 4),
        })

results_h7_df = pd.DataFrame(results_h7).sort_values(
    by=["f1_class_1", "accuracy"], ascending=False
).reset_index(drop=True)

print("=== Kết quả sau cải tiến H7 (Word-segmented) ===")
results_h7_df

=== Kết quả sau cải tiến H7 (Word-segmented) ===


,pipeline,vectorizer,model,accuracy,precision_class_1,recall_class_1,f1_class_1
0,TF-IDF + SVM [segmented],TF-IDF,SVM,0.9461,0.7429,0.6114,0.6707
1,TF-IDF + LogisticRegression [segmented],TF-IDF,LogisticRegression,0.9445,0.7587,0.5601,0.6445
2,BoW + LogisticRegression [segmented],BoW,LogisticRegression,0.9371,0.6548,0.6336,0.6440
3,BoW + SVM [segmented],BoW,SVM,0.9282,0.5951,0.6269,0.6106
4,BoW + MultinomialNB [segmented],BoW,MultinomialNB,0.8935,0.4537,0.9109,0.6057
5,TF-IDF + MultinomialNB [segmented],TF-IDF,MultinomialNB,0.9289,0.8582,0.2494,0.3865


In [15]:
baseline_seg = results_df[["vectorizer", "model", "accuracy",
                            "precision_class_1", "recall_class_1", "f1_class_1"]].copy()
baseline_seg["preprocessing"] = "Raw text"

after_seg = results_h7_df[["vectorizer", "model", "accuracy",
                             "precision_class_1", "recall_class_1", "f1_class_1"]].copy()
after_seg["preprocessing"] = "Word-segmented"

all_seg = pd.concat([baseline_seg, after_seg], ignore_index=True)

compare_h7 = all_seg.pivot_table(
    index=["vectorizer", "model"],
    columns="preprocessing",
    values=["accuracy", "f1_class_1", "recall_class_1", "precision_class_1"]
).round(4)

compare_h7.columns = ["_".join(col).replace(" ", "_") for col in compare_h7.columns]
compare_h7 = compare_h7.reset_index()

compare_h7["f1_delta"] = (
    compare_h7["f1_class_1_Word-segmented"] - compare_h7["f1_class_1_Raw_text"]
).round(4)

print("=== So sánh Raw text vs Word-segmented ===")
compare_h7[["vectorizer", "model",
            "f1_class_1_Raw_text", "f1_class_1_Word-segmented", "f1_delta",
            "accuracy_Raw_text",   "accuracy_Word-segmented"]].sort_values("f1_delta", ascending=False)


=== So sánh Raw text vs Word-segmented ===


,vectorizer,model,f1_class_1_Raw_text,f1_class_1_Word-segmented,f1_delta,accuracy_Raw_text,accuracy_Word-segmented
0,BoW,LogisticRegression,0.6134,0.6440,0.0306,0.9303,0.9371
2,BoW,SVM,0.5864,0.6106,0.0242,0.9224,0.9282
1,BoW,MultinomialNB,0.5897,0.6057,0.0160,0.8870,0.8935
3,TF-IDF,LogisticRegression,0.6408,0.6445,0.0037,0.9435,0.9445
5,TF-IDF,SVM,0.6691,0.6707,0.0016,0.9451,0.9461
4,TF-IDF,MultinomialNB,0.4684,0.3865,-0.0819,0.9328,0.9289


In [16]:
best_h4       = results_h4_df.iloc[0][["pipeline","accuracy","precision_class_1","recall_class_1","f1_class_1"]]
best_h7       = results_h7_df.iloc[0][["pipeline","accuracy","precision_class_1","recall_class_1","f1_class_1"]]

summary = pd.DataFrame([
    {"Phase": "H4: Bigram (1,2)",           **best_h4},
    {"Phase": "H7: Word-segmented",         **best_h7},
])

print("=== Tổng kết best pipeline mỗi giai đoạn ===")
summary

=== Tổng kết best pipeline mỗi giai đoạn ===


,Phase,pipeline,accuracy,precision_class_1,recall_class_1,f1_class_1
0,"H4: Bigram (1,2)","TF-IDF (1,2) + SVM",0.9478,0.7467,0.6336,0.6855
1,H7: Word-segmented,TF-IDF + SVM [segmented],0.9461,0.7429,0.6114,0.6707
